In [1]:
# Setup the Jupyter version of Dash
from dash import Dash
import sys

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
import dash

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
from src.sql_animal_shelter import SqlAnimalShelter
db = SqlAnimalShelter()
df = pd.DataFrame.from_records(db.read_all())

if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

#Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()


app.layout = html.Div([
    html.Img(src='data:image/png;base64,{}'.format(encoded_image), style={'width': '200px', 'display': 'block', 'margin': 'auto'}),
    
    html.Center(html.B(html.H1('Scott Moore | Last Updated: February 23, 2025'))),
    html.Hr(),
    
    #Filter buttons for rescue type
    html.Div(className='buttonRow',
             style={'display' : 'flex', 'justify-content': 'left'},
             children=[
                 html.Button(id='water-button', n_clicks=0, children='Water Rescue'),
                 html.Button(id='mountain-button', n_clicks=0, children='Mountain and Wilderness Rescue'),
                 html.Button(id='disaster-button', n_clicks=0, children='Disaster and Tracking'),
                 html.Button(id='reset-button', n_clicks=0, children='Reset')
             ]),
        
    html.Hr(),
    
    #Data Table
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         row_selectable="single",
                         selected_rows=[0],
                         page_action="native",
                         page_current=0,
                         page_size=10
                        ),

#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('water-button', 'n_clicks'),
               Input('mountain-button', 'n_clicks'),
               Input('disaster-button', 'n_clicks'),
               Input('reset-button', 'n_clicks')]
             )
def update_table(water_button, mountain_button, disaster_button, reset_button):
    ctx = dash.callback_context
    if not ctx or not ctx.triggered:
        return df.to_dict('records')

    trigger_id = ctx.triggered[0]['prop_id'].split('.')[0]

    if trigger_id == 'water-button':
        records = db.read_filtered("water")

    elif trigger_id == 'mountain-button':
        records = db.read_filtered("mountain")

    elif trigger_id == 'disaster-button':
        records = db.read_filtered("disaster")

    else:
        records = db.read_all()

    df_filtered = pd.DataFrame.from_records(records)
    return df_filtered.to_dict('records')

#Pie Chart callback
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "data")])
def update_pie_chart(data):
    df_filtered = pd.DataFrame.from_dict(data)
    
#if there are no results, display "No Data" in red
    if df_filtered.empty:
        return [html.P("No Data", style={"color": "red", "text-align": "center"})]
    
    return[
        dcc.Graph(
            figure=px.pie(df_filtered, names="breed", title="Breed Distribution")
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable

@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if not viewData:
        return [html.P("No data to map.", style={"textAlign": "center"})]

    dff = pd.DataFrame.from_dict(viewData)

    row = 0
    if index and len(index) > 0:
        row = index[0]

    # Guard against missing columns / bad values
    required = {"location_lat", "location_long", "breed", "name"}
    if not required.issubset(set(dff.columns)):
        return [html.P("Missing location columns in data.", style={"color": "red", "textAlign": "center"})]

    lat = dff.loc[row, "location_lat"]
    lon = dff.loc[row, "location_long"]

    if pd.isna(lat) or pd.isna(lon):
        return [html.P("Selected record has no location coordinates.", style={"color": "red", "textAlign": "center"})]

    breed = dff.loc[row, "breed"]
    name = dff.loc[row, "name"]

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(str(breed)),
                dl.Popup([html.H1("Animal Name"), html.P(str(name))])
            ])
        ])
    ]



app.run(debug=True, jupyter_mode="external")


Dash app running on http://127.0.0.1:8050/
